# ДЗ-16 (часть 1): LoRA fine-tuning чат-ассистента

Дообучаем небольшую instruct-модель под качественные диалоги методом **LoRA / QLoRA**
на сабсете датасета **lmsys/lmsys-chat-1m**.

## Зачем LoRA / PEFT
Полный fine-tuning LLM меняет все веса (миллиарды параметров) — это дорого по памяти и времени.
**PEFT** (Parameter-Efficient Fine-Tuning) обучает лишь малую добавку. **LoRA** вставляет в слои
внимания обучаемые низкоранговые матрицы `A·B` (ранг `r`), а исходные веса замораживает —
обучается ~0.1–1% параметров. **QLoRA** дополнительно квантует базовую модель в 4 бита.

## Модель выбирается автоматически под вашу видеокарту

| VRAM | Модель | Режим | Почему |
|---|---|---|---|
| < 6 ГБ | `SmolLM2-360M-Instruct` | fp16 + LoRA | словарь 49k → логиты влезают с запасом |
| ≥ 6 ГБ | `Qwen2.5-1.5B-Instruct` | 4-бит QLoRA | на T4/16 ГБ можно взять модель побольше |

**Ключевой момент: на 4 ГБ упирались не в размер модели, а в размер словаря.**
Главный потребитель памяти на шаге обучения — тензор логитов `batch × seq × vocab`,
который `transformers` в функции лосса ещё и копирует в fp32. У всех моделей Qwen
словарь 151936 токенов, поэтому Qwen2.5-**0.5B** упирается в тот же потолок, что и 1.5B
(замерено: те же ~3.2 ГБ и 157 ток/с). У SmolLM2 словарь 49152 — втрое меньше,
и именно это, а не число параметров, даёт выигрыш.

## Что было исправлено (причины `The Kernel crashed`)

1. **Тихий сброс в системную RAM — главная причина и крашей, и тормозов.**
   На Windows драйвер NVIDIA при нехватке VRAM не выдаёт ошибку, а незаметно вытесняет
   тензоры в обычную память через PCIe. Замер: `batch=2, seq_len=1024` "успешно" запрашивал
   **10.9 ГБ на карте с 4 ГБ**, и один шаг занимал **82 секунды**; затем ядро умирало.
   Исправление — `torch.cuda.set_per_process_memory_fraction(0.92)`: теперь вместо свопа
   мы получаем честный `CUDA out of memory` в traceback, который не убивает ядро.

2. **Логиты, а не веса модели** — см. выше; отсюда выбор модели с меньшим словарём
   и умеренное произведение `batch·seq`.

3. **Повторный запуск ячейки загружал вторую копию модели** поверх первой —
   теперь ячейка загрузки идемпотентна и сначала освобождает VRAM.

4. **`SFTTrainer` принудительно переводит QLoRA-адаптеры в bf16**, которого на Pascal нет
   (обучение падало на первом шаге). Отключить это нечем (peft#2889), поэтому адаптеры
   возвращаются в fp32 после создания трейнера. В fp16-режиме (SmolLM2) проблема не
   возникает вовсе — ещё одна причина не тащить bitsandbytes на старую карту.

## Замеры на Quadro P2000 (4 ГБ), полные прогоны

| Конфигурация | Пик VRAM | Пропускная способность |
|---|---|---|
| Было: Qwen-1.5B, `2 × 1024` | 10.9 ГБ «виртуально» | 82 с/шаг → **краш ядра** |
| Qwen-1.5B, 4-бит, `1 × 256`, accum 8 | 2.70 ГБ | ~135 ток/с |
| **SmolLM2-360M, fp16, `2 × 512`** | **1.19 ГБ** | **~259 ток/с** |

> ⚠️ **Про время обучения без иллюзий.** SmolLM2 обрабатывает токены примерно
> **в 1.9 раза быстрее**, но общее время прогона зависит ещё и от того, сколько токенов
> вы обрабатываете: `время = max_steps × accum × время_micro_batch`. При `accum=4`
> 60 шагов занимали те же ~16 минут, что и Qwen, просто модель успевала «увидеть»
> вчетверо больше данных. Поэтому в конфиге стоит `accum=2` — это даёт **~8 минут**
> на 60 шагов. Хотите ещё быстрее — уменьшайте `ACCUM` или `max_steps`
> (обе величины сокращают время линейно).

Запас по памяти теперь большой (1.2 ГБ из 4 ГБ), так что при желании можно, наоборот,
поднять `MAX_LEN` до 1024 или `BATCH` до 3–4. Если что-то всё же упадёт — берите
`HuggingFaceTB/SmolLM2-135M-Instruct` (ещё вдвое быстрее) или уменьшите `MAX_LEN`.

> ℹ️ `packing=True` (склейка коротких диалогов, убирает padding) мог бы дать ещё прирост,
> но без Flash Attention диалоги внутри блока начинают «видеть» друг друга. Flash Attention
> требует Ampere+, поэтому на Pascal packing намеренно выключен.

## Шаг 1. Установка (в Colab)
Раскомментируй и выполни в Colab. Локально с CUDA ставь из `requirements.txt`.

In [1]:
# !pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes

In [1]:
# =====================================================================
# ФИКС КРАША ЯДРА: аллокатор pyarrow  (должно стоять ДО импорта datasets)
# =====================================================================
# Симптом: "The Kernel crashed while executing code..." без traceback.
# Диагноз по дампу из %LOCALAPPDATA%\CrashDumps:
#     code=0xC0000005 (ACCESS_VIOLATION)
#     FAULTING MODULE: ...\site-packages\pyarrow\arrow.dll (offset 0xBC5431)
# То есть падает НЕ CUDA и НЕ нехватка видеопамяти, а нативный код pyarrow
# (на нём построен datasets). По умолчанию pyarrow 24 использует аллокатор
# mimalloc, и на этой связке (Windows + Python 3.13 + pyarrow 24 + numpy 2.x)
# он рушит процесс при работе с Arrow-таблицами.
#
# Проверено A/B на одинаковой нагрузке (20 циклов Dataset.from_list на нашем
# датасете) в одном долгоживущем процессе — как раз режим ядра Jupyter:
#     mimalloc (по умолчанию) -> Segmentation fault, exit 139
#     system                  -> все 20 итераций OK, exit 0
# В коротком скрипте краш часто «не виден»: работа успевает завершиться,
# код возврата 0, а дамп пишется уже при выгрузке процесса. Ядро Jupyter
# живёт долго — и умирает в середине сессии. Отсюда «падает без причины».
import os

os.environ["ARROW_DEFAULT_MEMORY_POOL"] = "system"

import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "CUDA-GPU не найден. Запусти в Colab с T4 (Runtime -> Change runtime type -> GPU)."
)

# ВАЖНО (Windows + NVIDIA): при нехватке VRAM драйвер по умолчанию НЕ падает с ошибкой,
# а незаметно вытесняет тензоры в системную RAM через PCIe ("system memory fallback").
# Обучение при этом не крашится сразу, но шаг замедляется в десятки раз, а затем ядро
# Jupyter умирает без внятного сообщения.
# Ставим жёсткий потолок: теперь вместо тихого свопа мы получим честный CUDA OOM,
# который видно в traceback и который не убивает ядро.
torch.cuda.set_per_process_memory_fraction(0.92)

props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
IS_PASCAL = props.major < 7          # P2000 = CC 6.1: нет bf16 и int8 tensor cores
LOW_VRAM = VRAM_GB < 6

# =====================================================================
# Выбор модели под объём видеопамяти
# =====================================================================
# Замеры на Quadro P2000 (4 ГБ); шаг = forward + backward + шаг оптимизатора:
#
#   модель                     словарь  режим  batch x seq  пик VRAM  скорость
#   Qwen2.5-1.5B-Instruct       151936  4-бит     1 x 256    2.70 ГБ  ~135 ток/с
#   Qwen2.5-0.5B-Instruct       151936  4-бит     1 x 512    3.19 ГБ  ~157 ток/с
#   SmolLM2-360M-Instruct        49152  fp16      2 x 512    1.19 ГБ  ~259 ток/с
#
# Почему SmolLM2, а не Qwen2.5-0.5B: у ВСЕХ моделей Qwen словарь 151936 токенов, а
# главный потребитель памяти на train-шаге — тензор логитов batch*seq*vocab
# (подробности в ячейке обучения). У 0.5B словарь тот же, что у 1.5B, поэтому узкое
# место никуда не девается — замер это подтвердил: те же ~3.2 ГБ.
# У SmolLM2 словарь втрое меньше — выигрыш даёт именно это, а не число параметров.
#
# Бонус: 360M обучается в обычном fp16 БЕЗ bitsandbytes, поэтому на Pascal отпадает
# целый класс проблем (4-битные ядра, принудительный bf16 для QLoRA-адаптеров).
if LOW_VRAM:
    MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    USE_4BIT = False                 # 360M спокойно помещается в fp16
else:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    USE_4BIT = True                  # на T4/16 ГБ берём модель побольше, в QLoRA

print(f"GPU: {props.name} | {VRAM_GB:.1f} ГБ | CC {props.major}.{props.minor}")
print(f"Режим: {'low-VRAM (<6 ГБ)' if LOW_VRAM else 'обычный'}, "
      f"{'Pascal — только fp16' if IS_PASCAL else 'поддерживается bf16'}")
print(f"Модель: {MODEL_NAME} ({'4-бит QLoRA' if USE_4BIT else 'fp16 + LoRA'})")


def free_vram(*objs):
    """Освободить VRAM: удалить объекты, собрать мусор, вернуть кэш аллокатора драйверу."""
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def vram_report(tag=""):
    used = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"[VRAM]{' ' + tag if tag else ''} занято {used:.2f} ГБ, пик {peak:.2f} ГБ / {VRAM_GB:.1f} ГБ")

C:\Users\user1\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: Quadro P2000 | 4.0 ГБ | CC 6.1
Режим: low-VRAM (<6 ГБ), Pascal — только fp16
Модель: HuggingFaceTB/SmolLM2-360M-Instruct (fp16 + LoRA)


## Шаг 2. Загрузка модели в 4-битном виде (QLoRA)

In [2]:
# =====================================================================
# Загрузка модели. Ячейка идемпотентна: повторный запуск не создаёт
# вторую копию модели в VRAM (раньше это добавляло ~1.1 ГБ и вело к OOM).
# =====================================================================

# Если модель уже в памяти (повторный прогон ячейки) — сначала освобождаем VRAM.
for _name in ("trainer", "model"):
    if _name in globals():
        print(f"Освобождаю предыдущий объект '{_name}' перед перезагрузкой...")
        globals().pop(_name)
free_vram()

load_kwargs = dict(
    dtype=torch.float16,             # Pascal не поддерживает bf16 — строго fp16
    # Явное device_map={"": 0} вместо "auto": при нехватке места "auto" молча раскидывает
    # часть слоёв на CPU/диск, и тогда каждый шаг тащит веса по PCIe (очень медленно).
    # Лучше честный OOM, чем незаметный оффлоад.
    device_map={"": 0},
)

if USE_4BIT:
    # bitsandbytes 8-bit (LLM.int8()) требует int8 tensor cores (CC >= 7.5, Turing+).
    # На Pascal быстрого пути нет, поэтому при квантовании берём именно 4-битный NF4:
    # он не требует int8-ядер.
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Загрузка токенизатора и модели {MODEL_NAME}...")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.config.use_cache = False   # обязательно при обучении с gradient checkpointing

vram_report("после загрузки модели")
print(f"✅ Модель загружена ({'4-бит NF4' if USE_4BIT else 'fp16'}), "
      f"словарь {model.config.vocab_size} токенов.")

Загрузка токенизатора и модели HuggingFaceTB/SmolLM2-360M-Instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1182.16it/s]


[VRAM] после загрузки модели занято 0.67 ГБ, пик 0.67 ГБ / 4.0 ГБ
✅ Модель загружена (fp16), словарь 49152 токенов.


## Шаг 3. Baseline ДО обучения
Сохраним ответ исходной модели на тестовый вопрос, чтобы потом сравнить с дообученной.

In [3]:
from contextlib import nullcontext


def chat(model, question: str, max_new_tokens: int = 200) -> str:
    """Генерация ответа по chat-шаблону модели, стабильная на CUDA и с квантованием."""
    messages = [{"role": "user", "content": question}]

    # Формируем правильный промпт со всеми разметками
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Токенизируем и явно отправляем на то же устройство, где находится первый слой модели
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # use_cache=True только для генерации (ускоряет вывод в разы).
    # Во время обучения он выключен — несовместим с gradient checkpointing.
    was_cache = model.config.use_cache
    was_training = model.training
    model.config.use_cache = True
    model.eval()

    # После обучения LoRA-адаптеры могут остаться в fp32, а база — в fp16, и обычный
    # forward падает с "expected mat1 and mat2 to have the same dtype".
    # autocast приводит типы к общему знаменателю прямо во время матричных операций.
    autocast = torch.autocast("cuda", dtype=torch.float16) if model.device.type == "cuda" \
        else nullcontext()
    try:
        with torch.inference_mode(), autocast:
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        prompt_length = inputs.input_ids.shape[1]
        answer = tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True).strip()
    finally:
        # Возвращаем прежний режим и освобождаем KV-кэш: иначе он остаётся в VRAM и
        # "съедает" память, которая нужна следующему шагу обучения.
        model.config.use_cache = was_cache
        model.train(was_training)
        free_vram()
    return answer


# Тестируем базовую модель
TEST_Q = "Объясни простыми словами, чем отличается обучение с учителем от обучения без учителя."
baseline_answer = chat(model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
vram_report("после baseline-генерации")

=== ДО fine-tuning ===
 Обучение с учителем от обучения без учителя, чем отличается обучение с учителем от обучения без учителя, чем отличается обучение с учителем от обучения без учителя.
[VRAM] после baseline-генерации занято 0.68 ГБ, пик 0.68 ГБ / 4.0 ГБ


## Шаг 4. Датасет: lmsys-chat-1m

`lmsys/lmsys-chat-1m` — **gated**: зайди на
[страницу датасета](https://huggingface.co/datasets/lmsys/lmsys-chat-1m), прими условия и
залогинься токеном (`HF_TOKEN`). Берём небольшой сабсет через streaming (не качаем весь 1M).

Если доступа к lmsys нет — функция автоматически переключится на открытый
`HuggingFaceH4/ultrachat_200k`.

In [4]:
import os
import json
from pathlib import Path
from itertools import islice

# --- ЗАЩИТА ОТ КРАША ЯДРА ---------------------------------------------------
# ARROW_DEFAULT_MEMORY_POOL действует только если выставлен ДО первого импорта
# pyarrow. Если эта ячейка запущена раньше самой первой (или ядро перезапускали
# и порядок сбился), аллокатор останется mimalloc — и ядро упадёт без traceback.
# Поэтому проверяем факт, а не намерение.
import pyarrow as pa

_pool = pa.default_memory_pool().backend_name
if _pool != "system":
    raise RuntimeError(
        f"pyarrow использует аллокатор '{_pool}', а не 'system'.\n"
        "На этой связке (Windows + Python 3.13 + pyarrow 24) mimalloc роняет ядро\n"
        "с ACCESS_VIOLATION в arrow.dll.\n"
        "ЧТО СДЕЛАТЬ: перезапустить ядро (Restart Kernel) и выполнить ячейки\n"
        "по порядку, начиная с самой первой — она выставляет нужную переменную."
    )
print(f"pyarrow memory pool: {_pool} ✅")

from datasets import load_dataset, Dataset
from huggingface_hub import login

if os.getenv("HF_TOKEN"):
    login(os.environ["HF_TOKEN"])

N_SAMPLES = 2000        # сабсет для демонстрации; увеличь для лучшего качества
MAX_TURNS = 6           # ограничим длину диалогов

# Стриминг 2000 диалогов с HF занимает ~80 секунд и повторяется при КАЖДОМ перезапуске ядра.
# Кэшируем результат на диск — повторный прогон ячейки становится мгновенным.
CACHE_FILE = Path(f"dialogues_cache_{N_SAMPLES}_{MAX_TURNS}.json")


def to_messages_lmsys(row):
    # в lmsys поле 'conversation' = [{'role': 'user'/'assistant', 'content': ...}, ...]
    return [{"role": m["role"], "content": m["content"]} for m in row["conversation"][:MAX_TURNS]]


def to_messages_ultrachat(row):
    return [{"role": m["role"], "content": m["content"]} for m in row["messages"][:MAX_TURNS]]


def download_chat_subset():
    try:
        ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)
        rows = [to_messages_lmsys(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из lmsys-chat-1m")
    except Exception as e:
        print(f"lmsys недоступен ({e}). Переключаюсь на ultrachat_200k.")
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        rows = [to_messages_ultrachat(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из ultrachat_200k")
    return rows


def load_chat_subset():
    if CACHE_FILE.exists():
        rows = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
        print(f"Взято из локального кэша {CACHE_FILE} — {len(rows)} диалогов (сеть не нужна)")
    else:
        rows = download_chat_subset()
        CACHE_FILE.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
        print(f"Сохранено в кэш {CACHE_FILE}")
    # оставляем только корректные диалоги (начинается с user, есть ответ ассистента)
    return [r for r in rows if len(r) >= 2 and r[0]["role"] == "user"]


dialogues = load_chat_subset()
print("Диалогов после фильтрации:", len(dialogues))
print("Пример диалога:", dialogues[0][:2])

pyarrow memory pool: system ✅
Взято из локального кэша dialogues_cache_2000_6.json — 2000 диалогов (сеть не нужна)
Диалогов после фильтрации: 2000
Пример диалога: [{'role': 'user', 'content': 'how can identity protection services help protect me against identity theft'}, {'role': 'assistant', 'content': "Identity protection services can help protect you against identity theft in several ways:\n\n1. Monitoring: Many identity protection services monitor your credit reports, public records, and other sources for signs of identity theft. If they detect any suspicious activity, they will alert you so you can take action.\n2. Credit freeze: Some identity protection services can help you freeze your credit, which makes it more difficult for thieves to open new accounts in your name.\n3. Identity theft insurance: Some identity protection services offer insurance that can help you recover financially if you become a victim of identity theft.\n4. Assistance: Many identity protection services off

## Шаг 5. Форматирование под chat-шаблон
SFTTrainer обучается на готовом тексте. Превращаем каждый диалог в строку через
`apply_chat_template` — так модель учится в том же формате, в котором её потом спрашивают.

In [5]:
def format_example(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [{"text": format_example(d)} for d in dialogues]
train_ds = Dataset.from_list(train_texts)
print("Обучающих примеров:", len(train_ds))
print("\n--- Пример отформатированного текста ---\n", train_ds[0]["text"][:400])

Обучающих примеров: 2000

--- Пример отформатированного текста ---
 <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
how can identity protection services help protect me against identity theft<|im_end|>
<|im_start|>assistant
Identity protection services can help protect you against identity theft in several ways:

1. Monitoring: Many identity protection services monitor your credit reports, public r


## Шаг 6. Конфиг LoRA и обучение (SFTTrainer)

Параметры LoRA:
- `r` — ранг добавок (8/16/32): больше → выразительнее и больше обучаемых параметров;
- `lora_alpha` — масштаб добавок (обычно 2·r);
- `target_modules` — в какие слои вставлять LoRA (проекции attention + MLP);
- `lora_dropout` — регуляризация.

In [6]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Перед обучением освобождаем всё, что осталось от baseline-генерации (KV-кэш и т.п.).
free_vram()

if USE_4BIT:
    model = prepare_model_for_kbit_training(
        model,
        # use_reentrant=False — современная реализация gradient checkpointing.
        # Со старой (True) PEFT-адаптеры иногда не получают градиент и обучение
        # молча "не идёт".
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    # prepare_model_for_kbit_training апкастит НЕ-квантованные веса в fp32. У Qwen2.5
    # embed_tokens (151936 x 1536) в fp32 занимает 890 МБ — 22% от 4 ГБ, причём lm_head
    # к нему привязан (tied weights). Мы его не обучаем, поэтому возвращаем в fp16.
    if LOW_VRAM:
        for _n, _p in model.named_parameters():
            if _p.dtype == torch.float32 and "embed_tokens" in _n:
                _p.data = _p.data.to(torch.float16)
                _p.requires_grad_(False)
        free_vram()
        vram_report("после fp16-эмбеддингов")
else:
    # Неквантованный путь (маленькая модель в fp16): prepare_model_for_kbit_training
    # здесь не нужен и даже вреден — он апкастит веса в fp32 и удваивает их размер.
    # Gradient checkpointing включаем вручную; enable_input_require_grads обязателен,
    # иначе при замороженной базе градиент не дойдёт до LoRA-адаптеров.
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# --- Подбор длины и батча под объём VRAM -----------------------------------------
# Главный потребитель памяти на train-шаге — НЕ веса модели, а тензор логитов:
# transformers в ForCausalLMLoss делает logits.float(), то есть держит fp16-логиты
# ПЛЮС их fp32-копию плюс градиент. Для батча B, длины L и словаря V это
# B*L*V*(2+4+4) байт. Именно поэтому размер СЛОВАРЯ важнее размера модели:
#   Qwen  (V=151936), B*L=2048 -> ~2.9 ГБ только на лосс (отсюда были краши);
#   SmolLM2 (V=49152), B*L=1024 -> ~0.5 ГБ — влезает с большим запасом.
#
# ВРЕМЯ ОБУЧЕНИЯ = max_steps * accum * (время одного micro-batch). Замер полного
# прогона на P2000: SmolLM2-360M даёт ~259 ток/с против ~135 ток/с у Qwen-1.5B
# (в 1.9 раза быстрее), но при accum=4 общее время осталось ~16 минут, потому что
# за шаг обрабатывается вчетверо больше токенов. Если нужен быстрый прогон ради
# демонстрации — уменьшайте ACCUM (это линейно сокращает время) или max_steps.
if LOW_VRAM:                      # 4 ГБ (P2000): замерено, пик всего ~1.2 ГБ из 4 ГБ
    MAX_LEN, BATCH, ACCUM = 512, 2, 2      # ~8 минут на 60 шагов
else:                             # T4 16 ГБ и подобные
    MAX_LEN, BATCH, ACCUM = 1024, 2, 4

print(f"Конфиг: max_length={MAX_LEN}, batch={BATCH}, accum={ACCUM} "
      f"(эффективный батч = {BATCH * ACCUM}, токенов на шаг = {BATCH * MAX_LEN * ACCUM})")

sft_config = SFTConfig(
    output_dir="lora-out",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    max_steps=60,
    learning_rate=2e-4,
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    # paged_adamw_8bit экономит память, но требует bitsandbytes. Без квантования
    # модель маленькая, и обычный adamw_torch работает быстрее и без лишней зависимости.
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",

    fp16=True,                         # Pascal не поддерживает bf16 — строго fp16
    bf16=False,

    max_length=MAX_LEN,
    # ВНИМАНИЕ про packing: он склеивает несколько диалогов в один блок и убирает padding,
    # НО без Flash Attention соседние диалоги внутри блока "видят" друг друга
    # (cross-contamination) — trl прямо предупреждает об этом при запуске.
    # Flash Attention требует Ampere+ и на Pascal недоступен, поэтому packing выключен:
    # корректность важнее скорости.
    packing=False,
    dataset_num_proc=1,                # на Windows многопроцессность datasets часто виснет
    dataloader_num_workers=0,          # 0 — на Windows spawn-воркеры только замедляют старт

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Периодически возвращаем кэш аллокатора драйверу — снижает фрагментацию,
    # из-за которой длинный прогон мог упасть по OOM ближе к концу.
    torch_empty_cache_steps=10,

    save_strategy="no",                # 60 шагов — промежуточные чекпоинты не нужны
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# --- ОБЯЗАТЕЛЬНО ДЛЯ PASCAL ------------------------------------------------------
# Для КВАНТОВАННОЙ модели SFTTrainer в конструкторе безусловно переводит все обучаемые
# веса в bf16 (рекомендация статьи QLoRA), и отключить это нечем: autocast_adapter_dtype
# для квантованных моделей пока не поддерживается (peft#2889).
# На Pascal аппаратного bf16 нет, и обучение падает на первом же шаге:
#   RuntimeError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'
# Проверяем именно compute capability, а НЕ torch.cuda.is_bf16_supported(): на P2000
# последняя возвращает True, так как учитывает медленную программную эмуляцию.
if IS_PASCAL:
    _fixed = 0
    for _p in trainer.model.parameters():
        if _p.requires_grad and _p.dtype == torch.bfloat16:
            _p.data = _p.data.to(torch.float32)
            _fixed += 1
    if _fixed:
        print(f"Pascal: {_fixed} обучаемых тензоров возвращены из bf16 в fp32")

trainer.model.print_trainable_parameters()

print("Запуск обучения...")
trainer.train()
vram_report("после обучения")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Конфиг: max_length=512, batch=2, accum=2 (эффективный батч = 4, токенов на шаг = 2048)


Building labels for train dataset (num_proc=1): 100%|██████████| 2000/2000 [00:16<00:00, 124.17 examples/s]


trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437
Запуск обучения...


Step,Training Loss
10,2.002660
20,1.769143
30,1.409388
40,1.412172
50,1.504728
60,1.437661


[VRAM] после обучения занято 0.79 ГБ, пик 1.19 ГБ / 4.0 ГБ


## Шаг 7. Сравнение ПОСЛЕ обучения
Тот же вопрос — но теперь отвечает модель с обученным LoRA-адаптером.

In [7]:
ft_answer = chat(trainer.model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
print("\n=== ПОСЛЕ fine-tuning ===\n", ft_answer)

=== ДО fine-tuning ===
 Обучение с учителем от обучения без учителя, чем отличается обучение с учителем от обучения без учителя, чем отличается обучение с учителем от обучения без учителя.

=== ПОСЛЕ fine-tuning ===
 Объясни простыми словами, чем отличается обучение с учителем от обучения без учителя.


## Шаг 8. Сохранение адаптера
Сохраняем только LoRA-адаптер (несколько МБ). В части 2 (`agent_demo.ipynb`) подгрузим его
поверх базовой модели.

In [8]:
# Имя папки выводим из имени модели: на 4 ГБ обучается SmolLM2-360M, на большой карте —
# Qwen2.5-1.5B, и адаптеры от разных баз путать нельзя (LoRA привязана к архитектуре).
ADAPTER_DIR = MODEL_NAME.split("/")[-1].lower() + "-lora-adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Адаптер сохранён в", ADAPTER_DIR)
print(f"Базовая модель для части 2: {MODEL_NAME}")
print("\nВ agent_demo.ipynb пропиши:")
print(f'  BASE_MODEL   = "{MODEL_NAME}"')
print(f'  ADAPTER_DIR  = "{ADAPTER_DIR}"')
print("  LOAD_ADAPTER = True")

# (опционально) слить адаптер в полную модель для удобного инференса:
# merged = trainer.model.merge_and_unload()
# merged.save_pretrained(MODEL_NAME.split("/")[-1].lower() + "-merged")

Адаптер сохранён в smollm2-360m-instruct-lora-adapter
Базовая модель для части 2: HuggingFaceTB/SmolLM2-360M-Instruct

В agent_demo.ipynb пропиши:
  BASE_MODEL   = "HuggingFaceTB/SmolLM2-360M-Instruct"
  ADAPTER_DIR  = "smollm2-360m-instruct-lora-adapter"
  LOAD_ADAPTER = True


## Выводы
- **LoRA/PEFT** позволил адаптировать модель, обучив доли процента параметров — это влезло
  в бесплатный GPU благодаря **QLoRA** (4-бит).
- Сравнение «до/после» на одном вопросе демонстрирует сдвиг стиля ответов к обучающим данным.
- Для реального качества: больше шагов (`max_steps`/эпохи), больше данных, валидация и подбор `r`.
- Дальше — `agent_demo.ipynb`: подключаем адаптер и даём модели **инструменты** (часть 2).